# 03 — Job Recommender (unsupervised)
TF-IDF the job corpus + resume → cosine similarity → top-N matching jobs.

## Objective

The objective of this notebook is to build a content-based job recommendation system that matches a candidate's resume with the most relevant job postings.

The workflow includes:

- Loading the processed job dataset
- Converting job descriptions into TF-IDF feature vectors
- Transforming a resume into the same feature space
- Computing cosine similarity between the resume and all jobs
- Ranking jobs based on similarity
- Returning the top matching job recommendations

In [2]:
# ============================================================
# Import Required Libraries
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import joblib

In [3]:
# ============================================================
# Load Job Dataset
# ============================================================

PROJECT_ROOT = Path("..")

PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

jobs_df = pd.read_csv(PROCESSED_DATA / "jobs_clean.csv")

print("Jobs dataset loaded successfully!")

display(jobs_df.head())

Jobs dataset loaded successfully!


,title,company,location,skills,description,experience,source,text
0,Walkin Data Entry Operator (night Shift),MM Media Pvt Ltd,Chennai,ITES,Job Description Send me Jobs like this Quali...,0 - 1 yrs,naukri,walkin data entry operator night shift ites jo...
1,Work Based Onhome Based Part Time.,find live infotech,Chennai,Marketing,Job Description Send me Jobs like this Quali...,0 - 0 yrs,naukri,work based onhome based part time. marketing j...
2,Pl/sql Developer - SQL,Softtech Career Infosystem Pvt. Ltd,Bengaluru,IT Software - Application Programming,Job Description Send me Jobs like this - as ...,4 - 8 yrs,naukri,pl sql developer sql it software application p...
3,Manager/ad/partner - Indirect Tax - CA,Onboard HRServices LLP,"Mumbai, Bengaluru, Kolkata, Chennai, Coimbator...",Accounts,Job Description Send me Jobs like this - Inv...,11 - 15 yrs,naukri,manager ad partner indirect tax ca accounts jo...
4,JAVA Technical Lead (6-8 yrs) -,Spire Technologies and Solutions Pvt. Ltd.,Bengaluru,IT Software - Application Programming,Job Description Send me Jobs like this Pleas...,6 - 8 yrs,naukri,java technical lead 6 8 yrs it software applic...


In [4]:
# ============================================================
# Dataset Overview
# ============================================================

print(f"Total Jobs       : {len(jobs_df)}")
print(f"Number of Columns: {jobs_df.shape[1]}")

display(jobs_df.head())

Total Jobs       : 136759
Number of Columns: 8


,title,company,location,skills,description,experience,source,text
0,Walkin Data Entry Operator (night Shift),MM Media Pvt Ltd,Chennai,ITES,Job Description Send me Jobs like this Quali...,0 - 1 yrs,naukri,walkin data entry operator night shift ites jo...
1,Work Based Onhome Based Part Time.,find live infotech,Chennai,Marketing,Job Description Send me Jobs like this Quali...,0 - 0 yrs,naukri,work based onhome based part time. marketing j...
2,Pl/sql Developer - SQL,Softtech Career Infosystem Pvt. Ltd,Bengaluru,IT Software - Application Programming,Job Description Send me Jobs like this - as ...,4 - 8 yrs,naukri,pl sql developer sql it software application p...
3,Manager/ad/partner - Indirect Tax - CA,Onboard HRServices LLP,"Mumbai, Bengaluru, Kolkata, Chennai, Coimbator...",Accounts,Job Description Send me Jobs like this - Inv...,11 - 15 yrs,naukri,manager ad partner indirect tax ca accounts jo...
4,JAVA Technical Lead (6-8 yrs) -,Spire Technologies and Solutions Pvt. Ltd.,Bengaluru,IT Software - Application Programming,Job Description Send me Jobs like this Pleas...,6 - 8 yrs,naukri,java technical lead 6 8 yrs it software applic...


## TF-IDF Feature Engineering

The job descriptions are converted into numerical feature vectors using TF-IDF.

This enables similarity comparison between resumes and job postings using cosine similarity.

In [5]:
# ============================================================
# Build TF-IDF Matrix for Jobs
# ============================================================

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

job_vectors = tfidf.fit_transform(jobs_df["text"])

print("TF-IDF Matrix Created Successfully!")
print("Shape :", job_vectors.shape)

TF-IDF Matrix Created Successfully!
Shape : (136759, 5000)


In [6]:
# ============================================================
# Save Recommendation Vectorizer
# ============================================================

PROJECT_ROOT = Path("..")
MODEL_DIR = PROJECT_ROOT / "models"

MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(tfidf, MODEL_DIR / "job_tfidf_vectorizer.pkl")

print("Recommendation TF-IDF Vectorizer Saved!")

Recommendation TF-IDF Vectorizer Saved!


In [18]:
# ============================================================
# Job Recommendation Function
# ============================================================

def recommend_jobs(resume_text, top_n=5):
    """
    Recommend the most relevant jobs for a given resume.
    """

    # Convert Resume to TF-IDF
    resume_vector = tfidf.transform([resume_text])

    # Calculate Cosine Similarity
    similarity_scores = cosine_similarity(
        resume_vector,
        job_vectors
    )

    # Flatten similarity scores
    similarity_scores = similarity_scores.flatten()

    # Get Top Matching Jobs
    top_indices = similarity_scores.argsort()[-top_n:][::-1]

    # Return Recommended Jobs
    recommendations = jobs_df.iloc[top_indices].copy()
    
    recommendations["Similarity Score"] = similarity_scores[top_indices]
    recommendations["Similarity Score"] = (
    recommendations["Similarity Score"]
    .round(3)
    )
    recommendations = recommendations.sort_values(
        by="Similarity Score",
        ascending=False
    )
    
    recommendations.insert(
        0,
        "Rank",
        range(1, len(recommendations) + 1)
    )
    
    return recommendations

In [19]:
# ============================================================
# Sample Resume
# ============================================================

sample_resume = """
Python
Machine Learning
TensorFlow
Deep Learning
SQL
Pandas
NumPy
Scikit-Learn
Data Analysis
"""

recommendations = recommend_jobs(sample_resume)

display(
    recommendations[
        [
            "Rank",
            "title",
            "company",
            "location",
            "experience",
            "Similarity Score"
        ]
    ]
)

,Rank,title,company,location,experience,Similarity Score
25969,1,Developer,Tata Consultancy Services,"New York, NY",Entry level,0.583
131363,2,Senior Machine Learning Engineer,Commit: AI Career Agents for Developers,United States,Mid-Senior level,0.563
129856,3,Lead Machine Learning Engineer,Confidential Jobs,United States,Mid-Senior level,0.561
76600,4,Machine Learning Engineer,eduPhoria.ai,"California, United States",NaN,0.555
123373,5,Applied Machine Learning Research Scientist,Coactive AI,"San Jose, CA",Mid-Senior level,0.549
